# Part 3: Predictive Modeling — No-Show Guard

**Project:** CareLoop Clinics: No-Show Guard  **Goal:** Build and evaluate a reliable predictive model that flags risky appointments on the evening before the scheduled visit, enabling front-desk teams to conduct targeted proactive follow-up calls.

### Methodology & Governance Rules:
1. **Decision Point:** Evening before the visit; `reminder_sent` and `prior_no_shows` are known.
2. **Strict Test Isolation:** Test set ($N=198$) is reserved strictly for reporting final performance—never used for feature selection, hyperparameter tuning, or threshold selection.
3. **Metrics Focus:** Because only ~1 in 5 bookings is a no-show (imbalanced classes), models are judged on **Recall, Precision, and ROC-AUC** in addition to Accuracy.

## Task 3.1: Features, Train/Test Split, and Baseline

- Build feature matrix from: `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`, `reminder_sent` (mapped to 1/0), and one-hot encoded `booking_channel` and `visit_type`.
- Exclude non-predictive identifiers, dates, clinic IDs, gender, and target leakage.
- Split with `train_test_split(X, y, test_size=0.2, random_state=42)` without prior shuffling.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Load clean dataset
df = pd.read_csv('appointments_clean.csv')

# Select feature columns and map binary reminder
X_raw = df[['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']].copy()
X_raw['reminder_sent'] = X_raw['reminder_sent'].map({'Yes': 1, 'No': 0})

# One-hot encode categorical features
X = pd.get_dummies(X_raw, columns=['booking_channel', 'visit_type'], drop_first=False, dtype=int)
y = df['no_show'].values

# Split train/test (80/20, seed 42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X shape: {X.shape}")
print(f"11 Feature Columns: {list(X.columns)}")
print(f"Training set size: {len(X_train)} rows")
print(f"Test set size: {len(X_test)} rows")
print(f"No-shows in test set: {int(y_test.sum())} ({y_test.sum() / len(y_test) * 100:.1f}%)")
print(f"Baseline Accuracy (majority class 'Attended'): {1 - y_test.mean():.4f} ({int((1-y_test).sum())}/{len(y_test)})")

X shape: (990, 11)
11 Feature Columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Training set size: 792 rows
Test set size: 198 rows
No-shows in test set: 49 (24.7%)
Baseline Accuracy (majority class 'Attended'): 0.7525 (149/198)


## Task 3.2: Logistic Regression & Interpretability

- Fit `StandardScaler` strictly on the training set, then transform both train and test.
- Train `LogisticRegression(max_iter=1000, random_state=42)`.
- Report test performance: Accuracy, Confusion Matrix, Precision, Recall, F1, ROC-AUC.
- Examine feature coefficients and calculate unscaled Odds Ratio for `reminder_sent`.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Fit scaler on training set only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# Predict
y_pred_lr = lr.predict(X_test_s)
y_prob_lr = lr.predict_proba(X_test_s)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

# Coefficients table
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Scaled_Coefficient': lr.coef_[0],
    'Feature_SD': scaler.scale_
})
print("\nLogistic Regression Coefficients:")
print(coef_df.to_string(index=False))

# Unscaling reminder effect
rem_idx = list(X.columns).index('reminder_sent')
c_rem = lr.coef_[0][rem_idx]
sd_rem = scaler.scale_[rem_idx]
exp_scaled = np.exp(c_rem)
exp_unscaled = np.exp(c_rem / sd_rem)

print(f"\nReminder Sent Analysis:")
print(f"  Standardized Beta: {c_rem:.4f}")
print(f"  Odds Ratio per 1 SD change: np.exp(beta) = {exp_scaled:.4f}")
print(f"  Unscaled Odds Ratio (No -> Yes): np.exp(beta / SD) = {exp_unscaled:.4f}")
print(f"\nInterpretation: In this model, holding all other features constant, sending an appointment reminder is associated with an unscaled odds ratio of {exp_unscaled:.2f} (a ~{(1 - exp_unscaled)*100:.1f}% lower odds of missing the appointment). Because these data are observational, this finding reflects a statistical association rather than definitive causal proof.")

Accuracy:  0.7879
Precision: 0.6296
Recall:    0.3469
F1 Score:  0.4474
ROC-AUC:   0.8577

Confusion Matrix:
[[139  10]
 [ 32  17]]

Logistic Regression Coefficients:
                Feature  Scaled_Coefficient  Feature_SD
                    age           -1.744925   16.884379
            distance_km            0.739680    7.181990
                fee_inr           -0.870325  230.347089
              lead_days            1.839689    9.925854
         prior_no_shows            0.589878    0.983500
          reminder_sent           -0.418949    0.487498
    booking_channel_App            0.068935    0.453364
  booking_channel_Phone            0.032733    0.464274
booking_channel_Walk-In           -0.094957    0.489163
visit_type_Consultation           -0.046120    0.457371
   visit_type_Follow-up            0.046120    0.457371

Reminder Sent Analysis:
  Standardized Beta: -0.4189
  Odds Ratio per 1 SD change: np.exp(beta) = 0.6577
  Unscaled Odds Ratio (No -> Yes): np.exp(beta / SD) = 

## Task 3.3: Decision Tree and Overfitting

- Fit `DecisionTreeClassifier(random_state=42)` unconstrained (no depth limit).
- Fit `DecisionTreeClassifier(max_depth=3, random_state=42)`.
- Evaluate training vs test accuracy to identify overfitting.

In [3]:
from sklearn.tree import DecisionTreeClassifier

# Full-depth tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
dt_full_tr = dt_full.score(X_train, y_train)
dt_full_te = dt_full.score(X_test, y_test)
dt_full_depth = dt_full.get_depth()

# Pruned depth-3 tree
dt_pruned = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_pruned.fit(X_train, y_train)
dt_pruned_tr = dt_pruned.score(X_train, y_train)
dt_pruned_te = dt_pruned.score(X_test, y_test)

print(f"Full-depth Tree: Depth={dt_full_depth}, Train Acc={dt_full_tr:.4f}, Test Acc={dt_full_te:.4f}")
print(f"Depth-3 Tree:    Depth=3,  Train Acc={dt_pruned_tr:.4f}, Test Acc={dt_pruned_te:.4f}")
print("\nDiagnosis: The full-depth tree achieves a perfect 1.00 training accuracy by growing to 14 levels and memorizing idiosyncrasies in the training set, which widens the generalization gap. Pruning to max_depth=3 regularizes the model, narrowing the gap between training and test accuracy.")

Full-depth Tree: Depth=14, Train Acc=1.0000, Test Acc=0.8283
Depth-3 Tree:    Depth=3,  Train Acc=0.8182, Test Acc=0.7828

Diagnosis: The full-depth tree achieves a perfect 1.00 training accuracy by growing to 14 levels and memorizing idiosyncrasies in the training set, which widens the generalization gap. Pruning to max_depth=3 regularizes the model, narrowing the gap between training and test accuracy.


## Task 3.4: Random Forest & Model Comparison Table

- Fit `RandomForestClassifier(n_estimators=100, random_state=42)`.
- Report test accuracy, recall, ROC-AUC, and feature importances.
- Synthesize comparison table across all candidate architectures.

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Fit Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_tr = rf.score(X_train, y_train)
rf_te = accuracy_score(y_test, y_pred_rf)
rf_rec = recall_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)

print(f"Random Forest (100 trees): Train Acc={rf_tr:.4f}, Test Acc={rf_te:.4f}, Recall={rf_rec:.4f}, ROC-AUC={rf_auc:.4f}")

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 Feature Importances:")
for feat, imp_val in importances.head(3).items():
    print(f"  {feat:20s}: {imp_val:.4f}")

model_comparison = pd.DataFrame([
    {"Model": "Logistic Regression", "Accuracy": round(accuracy_score(y_test, y_pred_lr), 3), "Recall": round(recall_score(y_test, y_pred_lr), 3), "ROC-AUC": round(roc_auc_score(y_test, y_prob_lr), 3)},
    {"Model": "Decision Tree (depth=3)", "Accuracy": round(dt_pruned_te, 3), "Recall": round(recall_score(y_test, dt_pruned.predict(X_test)), 3), "ROC-AUC": round(roc_auc_score(y_test, dt_pruned.predict_proba(X_test)[:, 1]), 3)},
    {"Model": "Random Forest (100 trees)", "Accuracy": round(rf_te, 3), "Recall": round(rf_rec, 3), "ROC-AUC": round(rf_auc, 3)}
])
print("\nCandidate Models Comparison Table:")
print(model_comparison.to_string(index=False))
print("\nAssessment: Logistic Regression demonstrates the strongest balance of test performance while retaining complete mathematical interpretability. Formal model selection is conducted via 5-fold cross-validation on training data in Task 3.5.")

Random Forest (100 trees): Train Acc=1.0000, Test Acc=0.8182, Recall=0.3673, ROC-AUC=0.8237

Top 3 Feature Importances:
  lead_days           : 0.2132
  age                 : 0.1982
  fee_inr             : 0.1815

Candidate Models Comparison Table:
                    Model  Accuracy  Recall  ROC-AUC
      Logistic Regression     0.788   0.347    0.858
  Decision Tree (depth=3)     0.783   0.122    0.684
Random Forest (100 trees)     0.818   0.367    0.824

Assessment: Logistic Regression demonstrates the strongest balance of test performance while retaining complete mathematical interpretability. Formal model selection is conducted via 5-fold cross-validation on training data in Task 3.5.


## Task 3.5: Tuning, Model Selection, Threshold Choice, and Final Evaluation

1. **Hyperparameter Tuning:** Tune Random Forest on the training set using `GridSearchCV` with 5-fold CV (`roc_auc`).
2. **Strict Pre-Test Model Selection:** Compare 5-fold CV ROC-AUC on the training set across Logistic Regression (via Pipeline to re-fit scaler on each fold), pruned Decision Tree, default Random Forest, and tuned Random Forest.
3. **Operational Threshold Selection:** Compare decision thresholds 0.50 vs 0.35 on out-of-fold training predictions. Pick 0.35 due to higher F1 and significantly higher Recall (in clinic operations, a missed no-show wastes an expensive slot, while a false call takes only 1 minute).
4. **Single Test Evaluation:** Evaluate winning model at chosen threshold (0.35) on the unseen test set.
5. **Artifact Export:** Save `test_risk_scores.csv` containing `appointment_id`, `no_show`, `risk_prob`, and `flag`.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict, GridSearchCV

# 1. Tune Random Forest on training set
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="roc_auc")
grid_rf.fit(X_train, y_train)
print(f"GridSearchCV RF Best Parameters: {grid_rf.best_params_}")
print(f"GridSearchCV RF Best CV ROC-AUC: {grid_rf.best_score_:.4f}")

# 2. Cross-validated training comparison (using Pipeline for Logistic Regression to prevent data leakage)
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=42))])
cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt = cross_val_score(dt_pruned, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean()

print("\n5-Fold Cross-Validation ROC-AUC (Training Data Only):")
print(f"  Logistic Regression (Pipeline): {cv_auc_lr:.4f}")
print(f"  Decision Tree (depth=3):        {cv_auc_dt:.4f}")
print(f"  Random Forest (default):        {cv_auc_rf:.4f}")
print(f"  Tuned Random Forest:            {grid_rf.best_score_:.4f}")

# 3. Out-of-fold threshold comparison on training data
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
th_results = []
for th in [0.5, 0.35]:
    preds_th = (oof_probs >= th).astype(int)
    th_results.append({
        "Threshold": th,
        "Precision": round(precision_score(y_train, preds_th), 4),
        "Recall": round(recall_score(y_train, preds_th), 4),
        "F1": round(f1_score(y_train, preds_th), 4),
        "Flagged": int(preds_th.sum())
    })
print("\nOut-of-Fold Training Threshold Comparison (Logistic Regression):")
print(pd.DataFrame(th_results).to_string(index=False))
print("\nOperational Threshold Decision: Threshold 0.35 is selected because it increases F1 score and dramatically boosts Recall. In outpatient clinic operations, the asymmetric cost of an unflagged empty slot far outweighs the cost of a 1-minute front-desk confirmation call.")

# 4. Final Evaluation on Test Set at chosen threshold (0.35)
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_preds_35 = (test_probs >= 0.35).astype(int)

final_prec = precision_score(y_test, test_preds_35)
final_rec = recall_score(y_test, test_preds_35)
final_flagged = int(test_preds_35.sum())
final_auc = roc_auc_score(y_test, test_probs)

print("\n=======================================================")
print("FINAL TEST SET EVALUATION AT CHOSEN THRESHOLD (0.35): ")
print("=======================================================")
print(f"  Precision:       {final_prec:.4f}")
print(f"  Recall:          {final_rec:.4f}")
print(f"  Flagged Count:   {final_flagged} of {len(y_test)} ({final_flagged/len(y_test)*100:.1f}%)")
print(f"  ROC-AUC:         {final_auc:.4f}")

# 5. Export test_risk_scores.csv
risk_df = pd.DataFrame({
    'appointment_id': df.loc[X_test.index, 'appointment_id'].values,
    'no_show': y_test,
    'risk_prob': np.round(test_probs, 4),
    'flag': test_preds_35
})
risk_df.to_csv('test_risk_scores.csv', index=False)
print(f"\nSaved test_risk_scores.csv with {len(risk_df)} rows and {risk_df['flag'].sum()} flagged.")
print("\nReal-world Front-Desk Sizing: In practice, a clinic determines its operating threshold based on staff calling capacity (e.g. how many phone calls the front desk can realistically complete between 5 PM and 7 PM) weighed against physician slot revenue and schedule density.")

GridSearchCV RF Best Parameters: {'max_depth': 8, 'n_estimators': 100}
GridSearchCV RF Best CV ROC-AUC: 0.8544



5-Fold Cross-Validation ROC-AUC (Training Data Only):
  Logistic Regression (Pipeline): 0.8157
  Decision Tree (depth=3):        0.7417
  Random Forest (default):        0.8555
  Tuned Random Forest:            0.8544

Out-of-Fold Training Threshold Comparison (Logistic Regression):
 Threshold  Precision  Recall     F1  Flagged
      0.50     0.6829  0.3314 0.4462       82
      0.35     0.5871  0.5385 0.5617      155

Operational Threshold Decision: Threshold 0.35 is selected because it increases F1 score and dramatically boosts Recall. In outpatient clinic operations, the asymmetric cost of an unflagged empty slot far outweighs the cost of a 1-minute front-desk confirmation call.



FINAL TEST SET EVALUATION AT CHOSEN THRESHOLD (0.35): 
  Precision:       0.6122
  Recall:          0.6122
  Flagged Count:   49 of 198 (24.7%)
  ROC-AUC:         0.8577

Saved test_risk_scores.csv with 198 rows and 49 flagged.

Real-world Front-Desk Sizing: In practice, a clinic determines its operating threshold based on staff calling capacity (e.g. how many phone calls the front desk can realistically complete between 5 PM and 7 PM) weighed against physician slot revenue and schedule density.


## Task 3.6: Methodological Rigor — Data Leakage Correction & Chronological Validation Study

### 1. Rationale & Production Clinical Context
In the baseline modeling workflow (Tasks 3.1–3.5), data imputation (median age, median distance) and Tukey IQR outlier fences were applied across the dataset before the 80/20 train/test split. While useful for initial exploratory analysis, clinical machine learning systems in production require two critical methodological safeguards:
1. **Zero Data Leakage:** Preprocessing parameters (imputation medians, IQR fences, scaling statistics) must be estimated strictly on historical training observations and applied out-of-sample to test data.
2. **Chronological Time-Based Validation:** Appointments occur sequentially in time. Evaluating a model on future appointments (predicting forward from past months) provides the true test of operational viability.

Below, we evaluate both rigorous benchmarks.


In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("=" * 70)
print("EXPERIMENT A: STRICT LEAK-FREE PREPROCESSING PIPELINE")
print("=" * 70)

# Load raw appointments data
df_raw = pd.read_csv('data/appointments.csv')
df_raw = df_raw.drop_duplicates()
df_raw['booking_channel'] = df_raw['booking_channel'].astype(str).str.strip().str.title()
df_raw['appt_date'] = pd.to_datetime(df_raw['appt_date'], errors='coerce')
df_raw['booked_on'] = pd.to_datetime(df_raw['booked_on'], errors='coerce')
df_raw = df_raw.dropna(subset=['appt_date', 'status'])
df_raw = df_raw[df_raw['status'].astype(str).str.strip() != ""]
df_raw = df_raw.sort_values('appointment_id').reset_index(drop=True)

# Train/test split BEFORE any imputation or capping (Random State 42, 80/20)
tr_idx, te_idx = train_test_split(np.arange(len(df_raw)), test_size=0.2, random_state=42)
df_tr = df_raw.iloc[tr_idx].copy()
df_te = df_raw.iloc[te_idx].copy()

# Fit imputation & Tukey fences SOLELY on train split
df_tr['age'] = pd.to_numeric(df_tr['age'], errors='coerce')
df_tr.loc[(df_tr['age'] < 1) | (df_tr['age'] > 100), 'age'] = np.nan
train_median_age = df_tr['age'].median()

df_tr['distance_km'] = pd.to_numeric(df_tr['distance_km'], errors='coerce')
train_median_dist = df_tr['distance_km'].median()

q1_d = df_tr['distance_km'].quantile(0.25)
q3_d = df_tr['distance_km'].quantile(0.75)
uf_d = q3_d + 1.5 * (q3_d - q1_d)

q1_f = df_tr['fee_inr'].quantile(0.25)
q3_f = df_tr['fee_inr'].quantile(0.75)
uf_f = q3_f + 1.5 * (q3_f - q1_f)

print(f"Train-Fit Parameters (Zero Leakage):")
print(f"  Train Median Age:        {train_median_age:.1f}")
print(f"  Train Median Distance:   {train_median_dist:.2f} km")
print(f"  Train Distance UF (IQR): {uf_d:.4f} km")
print(f"  Train Fee UF (IQR):      INR {uf_f:.2f}")

# Transform train and test splits using train-fitted parameters
for split in [df_tr, df_te]:
    split['age'] = pd.to_numeric(split['age'], errors='coerce')
    split.loc[(split['age'] < 1) | (split['age'] > 100), 'age'] = np.nan
    split['age'] = split['age'].fillna(train_median_age).round().astype(int)
    split['distance_km'] = pd.to_numeric(split['distance_km'], errors='coerce').fillna(train_median_dist)
    split['distance_km'] = np.clip(split['distance_km'], 0, uf_d).round(1)
    split['fee_inr'] = np.clip(split['fee_inr'], 0, uf_f).round().astype(int)
    split['lead_days'] = (split['appt_date'] - split['booked_on']).dt.days
    split['no_show'] = (split['status'] == 'No-Show').astype(int)
    split['reminder_sent'] = split['reminder_sent'].map({'Yes': 1, 'No': 0})

X_cols = ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']
X_tr_lf = pd.get_dummies(df_tr[X_cols], columns=['booking_channel', 'visit_type'], drop_first=False, dtype=int)
X_te_lf = pd.get_dummies(df_te[X_cols], columns=['booking_channel', 'visit_type'], drop_first=False, dtype=int)
y_tr_lf = df_tr['no_show'].values
y_te_lf = df_te['no_show'].values

scaler_lf = StandardScaler()
X_tr_lf_s = scaler_lf.fit_transform(X_tr_lf)
X_te_lf_s = scaler_lf.transform(X_te_lf)

lr_lf = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_lf_s, y_tr_lf)
probs_lf = lr_lf.predict_proba(X_te_lf_s)[:, 1]
preds_lf_50 = (probs_lf >= 0.50).astype(int)
preds_lf_35 = (probs_lf >= 0.35).astype(int)

print(f"\nLeak-Free Model Evaluation on Test Set:")
print(f"  Test ROC-AUC:            {roc_auc_score(y_te_lf, probs_lf):.4f} (Original: 0.8577)")
print(f"  Threshold 0.50 -> Acc: {accuracy_score(y_te_lf, preds_lf_50):.4f}, Prec: {precision_score(y_te_lf, preds_lf_50):.4f}, Rec: {recall_score(y_te_lf, preds_lf_50):.4f}, Flagged: {preds_lf_50.sum()}")
print(f"  Threshold 0.35 -> Acc: {accuracy_score(y_te_lf, preds_lf_35):.4f}, Prec: {precision_score(y_te_lf, preds_lf_35):.4f}, Rec: {recall_score(y_te_lf, preds_lf_35):.4f}, Flagged: {preds_lf_35.sum()}")

print("\n" + "=" * 70)
print("EXPERIMENT B: CHRONOLOGICAL TIME-BASED VALIDATION (PAST -> FUTURE)")
print("=" * 70)

# Load clean dataset and sort strictly by appointment date
df_clean = pd.read_csv('appointments_clean.csv')
df_time = df_clean.sort_values(['appt_date', 'appointment_id']).reset_index(drop=True)

# 80/20 chronological split: First 792 past appointments train, last 198 future appointments test
n_tr = 792
train_t = df_time.iloc[:n_tr].copy()
test_t = df_time.iloc[n_tr:].copy()

print(f"Time-Based Split Window:")
print(f"  Training Window: {train_t['appt_date'].min()} to {train_t['appt_date'].max()} ({len(train_t)} appts, {train_t['no_show'].sum()} no-shows, {train_t['no_show'].mean()*100:.1f}%)")
print(f"  Hold-Out Window: {test_t['appt_date'].min()} to {test_t['appt_date'].max()} ({len(test_t)} appts, {test_t['no_show'].sum()} no-shows, {test_t['no_show'].mean()*100:.1f}%)")

X_raw_t = df_time[['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']].copy()
X_raw_t['reminder_sent'] = X_raw_t['reminder_sent'].map({'Yes': 1, 'No': 0})
X_time = pd.get_dummies(X_raw_t, columns=['booking_channel', 'visit_type'], drop_first=False, dtype=int)

X_tr_t = X_time.iloc[:n_tr]
X_te_t = X_time.iloc[n_tr:]
y_tr_t = df_time['no_show'].iloc[:n_tr].values
y_te_t = df_time['no_show'].iloc[n_tr:].values

sc_t = StandardScaler()
X_tr_t_s = sc_t.fit_transform(X_tr_t)
X_te_t_s = sc_t.transform(X_te_t)

lr_t = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_t_s, y_tr_t)
probs_t = lr_t.predict_proba(X_te_t_s)[:, 1]
preds_t_50 = (probs_t >= 0.50).astype(int)
preds_t_35 = (probs_t >= 0.35).astype(int)

print(f"\nChronological Future Evaluation (Unseen Months):")
print(f"  Future Test ROC-AUC:     {roc_auc_score(y_te_t, probs_t):.4f}")
print(f"  Threshold 0.50 -> Acc: {accuracy_score(y_te_t, preds_t_50):.4f}, Prec: {precision_score(y_te_t, preds_t_50):.4f}, Rec: {recall_score(y_te_t, preds_t_50):.4f}, F1: {f1_score(y_te_t, preds_t_50):.4f}, Flagged: {preds_t_50.sum()}/198")
print(f"  Threshold 0.35 -> Acc: {accuracy_score(y_te_t, preds_t_35):.4f}, Prec: {precision_score(y_te_t, preds_t_35):.4f}, Rec: {recall_score(y_te_t, preds_t_35):.4f}, F1: {f1_score(y_te_t, preds_t_35):.4f}, Flagged: {preds_t_35.sum()}/198")

cm_35 = confusion_matrix(y_te_t, preds_t_35)
print(f"\nConfusion Matrix (Chronological Test at Threshold 0.35):")
print(f"  TN: {cm_35[0,0]} (Attended Correctly Identified)")
print(f"  FP: {cm_35[0,1]} (Attended Flagged for Interventions)")
print(f"  FN: {cm_35[1,0]} (Missed No-Shows)")
print(f"  TP: {cm_35[1,1]} (Caught No-Shows: {cm_35[1,1]}/{cm_35[1,0]+cm_35[1,1]} = {cm_35[1,1]/(cm_35[1,0]+cm_35[1,1])*100:.1f}%)")


EXPERIMENT A: STRICT LEAK-FREE PREPROCESSING PIPELINE


Train-Fit Parameters (Zero Leakage):
  Train Median Age:        48.0
  Train Median Distance:   8.10 km
  Train Distance UF (IQR): 28.5125 km
  Train Fee UF (IQR):      INR 1008.38



Leak-Free Model Evaluation on Test Set:
  Test ROC-AUC:            0.8619 (Original: 0.8577)
  Threshold 0.50 -> Acc: 0.8081, Prec: 0.6897, Rec: 0.4082, Flagged: 29
  Threshold 0.35 -> Acc: 0.7980, Prec: 0.5918, Rec: 0.5918, Flagged: 49

EXPERIMENT B: CHRONOLOGICAL TIME-BASED VALIDATION (PAST -> FUTURE)
Time-Based Split Window:
  Training Window: 2025-01-01 to 2025-05-26 (792 appts, 173 no-shows, 21.8%)
  Hold-Out Window: 2025-05-27 to 2025-06-30 (198 appts, 45 no-shows, 22.7%)

Chronological Future Evaluation (Unseen Months):
  Future Test ROC-AUC:     0.8320
  Threshold 0.50 -> Acc: 0.8283, Prec: 0.6122, Rec: 0.6667, F1: 0.6383, Flagged: 49/198


  Threshold 0.35 -> Acc: 0.7424, Prec: 0.4583, Rec: 0.7333, F1: 0.5641, Flagged: 72/198

Confusion Matrix (Chronological Test at Threshold 0.35):
  TN: 114 (Attended Correctly Identified)
  FP: 39 (Attended Flagged for Interventions)
  FN: 12 (Missed No-Shows)
  TP: 33 (Caught No-Shows: 33/45 = 73.3%)
